# Visualization: GeoZarr in the browser with EOxElements

This notebook shows **GeoZarr** data cubes on interactive maps, straight from object storage (S3) in a Jupyter notebook.

**What is GeoZarr?** [Zarr](https://zarr.dev/) stores n-dimensional arrays (e.g. `time × depth × lat × lon`) as many small compressed *chunks*. [GeoZarr](https://geozarr.org/) adds metadata conventions so that a map viewer knows where the data is on Earth and how to show it fast. Each convention is stored as attributes in the store's `zarr.json`:

| Convention | Attributes | What the viewer uses it for |
|---|---|---|
| `proj:` | `proj:code` (e.g. `EPSG:4326`) | The coordinate reference system (CRS) of the grid |
| `spatial:` | `spatial:bbox`, `spatial:transform`, `spatial:dimensions` | Where the pixels are, and which array dimensions are x/y |
| `multiscales` | `layout: [{asset: "0"}, {asset: "1"}, ...]` | A pyramid of resolution levels (`0` = full resolution), like overviews in a COG |

This is what the conversion notebook (`2_geozarr_conversion.ipynb`) produces.

**How it is rendered.** No tile server is involved. The map is [`<eox-map>`](https://github.com/EOX-A/EOxElements) (OpenLayers under the hood), wrapped for Python by [`ipyeoxelements`](https://github.com/EOX-A/EOxElements-Jupyter). It runs in your browser and:

1. reads the store metadata (`zarr.json`) to find the CRS, extent, resolution levels, variables and dimensions;
2. for every map tile on screen, picks the pyramid level closest to the current zoom and fetches only the chunks it needs, using HTTP range requests;
3. colours the raw values on the GPU (a `WebGLTile` layer) using a **style expression**. Changing colours or value ranges is instant and needs no new data.

The Python kernel only *describes* the map. All data loading and drawing happens in the browser, so the store must allow cross-origin (CORS) requests.

**Widgets used**

- `EOxMap` → `<eox-map>`: the map
- `EOxLayercontrol` → `<eox-layercontrol>`: layer list, plus a form to pick the variable and the min/max colour range
- `EOxTimecontrol` → `<eox-timecontrol>`: date picker / slider that moves the map through the cube's `time` dimension

**Before you run:** `timecontrol.js` and `options.json` must be next to this notebook. Run the cells **in order**: the polar map (section 2) reuses the map from section 1.

## 1. Mediterranean biophysics cube (EPSG:4326)

The store [`ocean-med-biophysics.zarr`](https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/med_cubes/ocean-med-biophysics.zarr/zarr.json) is a daily Mediterranean cube: 2404 time steps from 2016-01-01, on a `lat`/`lon` grid in `EPSG:4326`, with 3 resolution levels (`0`, `1`, `2`). Some of its variables:

| Variable | Dimensions | Description | Units |
|---|---|---|---|
| `mlotst` | time, lat, lon | Mixed layer depth | m |
| `ZEU` | time, lat, lon | Euphotic depth | m |
| `to` | time, depth, lat, lon | Temperature | °C |
| `so` | time, depth, lat, lon | Salinity | PSU |
| `PP` | time, depth, lat, lon | Primary production | mg m-3 d-1 |

**How the layer definition maps onto the store**

- `source.type: "GeoZarr"` + `url`: the root of the store. Everything else (CRS, extent, levels) is read from its metadata.
- `source.bands: []`: no variable chosen. `<eox-map>` lists the store's data variables, skipping coordinates like `lat`, `lon`, `time` and `depth`, and shows the first one. Use e.g. `"bands": ["to"]` to start on a specific variable.
- **Dimensions:** the `time` dimension is turned into a list of dates for the time control. Any other dimension (here `depth`) is set to index `0`, the top level.
- `properties.layerConfig` with `autofill: True`: the layer control builds a form for this layer. It has a **Variable** dropdown (names and units from the store attributes) and **Min / Max** inputs, which are wired to the style variables `min` and `max`. Keep those names in the style for the form to work.

**Reading the style expression.** The source hands each pixel to the GPU as bands:

- `["band", 1]` is the value of the selected variable;
- `["band", 2]` is a *data mask* the source adds when the variable has a fill value (here `NaN`): `1` where there is data, `0` where there is none (land, missing values).

So `["interpolate", ["linear"], ["band", 1], stop1, color1, stop2, color2, ...]` blends between four colours spread evenly from `min` to `max`. Each colour uses `["band", 2]` as its alpha (opacity), so pixels without data become transparent and the basemap shows through.

In [1]:
# ==============================================================================
# 1. Mediterranean biophysics cube: GeoZarr on a Web Mercator map
# ==============================================================================

import importlib
import ipyeoxelements
import ipyeoxelements.generated

# The time control is customised below by editing EOxTimecontrol's JavaScript template.
# When this cell runs again, reload the module first so the template is clean again
# (otherwise the child components would be added twice).
if "eox-timecontrol-date" in ipyeoxelements.EOxTimecontrol._esm_template:
    importlib.reload(ipyeoxelements.generated)
    importlib.reload(ipyeoxelements)

from ipyeoxelements import EOxMap, EOxLayercontrol, EOxTimecontrol
from ipywidgets import HBox, Layout, VBox




# Root URL of the GeoZarr store. The browser reads `zarr.json` here, then fetches only the
# chunks it needs from the resolution levels `0/`, `1/`, `2/`.
data = "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/med_cubes/ocean-med-biophysics.zarr"


# ------------------------------------------------------------------------------
# Layer Configurations (EOxElements layer JSON, written as Python dicts)
# ------------------------------------------------------------------------------
# Layers are listed bottom to top: the OSM basemap first, the GeoZarr data on top of it.
layers = [
    {"type": "Tile", "source": {"type": "OSM"}},
    {
        # WebGLTile: raw data values are coloured on the GPU by the `style` below
        "type": "WebGLTile",
        "properties": {
            # timecontrol.js looks the layer up by this id
            "id": "sea-ice-cube",
            "title": "Mediterranean Biophysics GeoZarr",
            # Show this layer and its tools expanded in the layer control
            "layerControlExpand": True,
            "layerControlToolsExpand": True,
            # Layer control "config" tool: `autofill` builds a form from the store metadata,
            # with a Variable dropdown and Min / Max inputs bound to style variables `min`/`max`
            "layerConfig": {
                "type": "style",
                "style": True,
                "autofill": True,
            },
        },
        "source": {
            "type": "GeoZarr",
            # Fetch chunks with CORS so WebGL can read the pixel values
            "crossOrigin": "anonymous",
            "url": data,
            # Empty list: show the first data variable found in the store
            # (e.g. ["to"] to start on temperature instead)
            "bands": [],
        },
        # Style expression run per pixel on the GPU:
        #   ["band", 1] = value of the selected variable
        #   ["band", 2] = data mask (1 = data, 0 = fill value / no data)
        "style": {
            # Colour range; also the initial values of the Min / Max inputs in the layer control
            "variables": {
                "min": 0,
                "max": 40,
            },
            # Linear ramp through 4 colours spread evenly from min to max
            # (cyan -> light yellow -> orange -> dark red). The 4th value of each
            # ["color", r, g, b, a] is the alpha: the data mask makes no-data pixels transparent.
            "color": [
                "interpolate",
                ["linear"],
                ["band", 1],
                # stop 1: value == min
                ["var", "min"],
                ["color", 0, 210, 255, ["band", 2]],
                # stop 2: min + 1/3 of the range
                [
                    "+",
                    ["var", "min"],
                    ["*", ["-", ["var", "max"], ["var", "min"]], 0.3333333333333333],
                ],
                ["color", 254, 217, 118, ["band", 2]],
                # stop 3: min + 2/3 of the range
                [
                    "+",
                    ["var", "min"],
                    ["*", ["-", ["var", "max"], ["var", "min"]], 0.6666666666666666],
                ],
                ["color", 253, 141, 60, ["band", 2]],
                # stop 4: value == max
                ["var", "max"],
                ["color", 189, 0, 38, ["band", 2]],
            ],
        },
    },
]


# EOxTimecontrol on its own only renders the <eox-timecontrol> container. timecontrol.js adds
# the date display, the date picker and the slider (as in the EOxElements Storybook example),
# and wires them to the map: when the date changes it finds the GeoZarr layer, looks up the
# matching index of the store's `time` dimension, and loads that slice
# (`source.updateDimensions({time: index})`).
time_control_file = "./timecontrol.js"
with open(time_control_file, "r") as f:
    time_control_element = f.read()

if "eox-timecontrol-date" not in EOxTimecontrol._esm_template:
    EOxTimecontrol._esm_template = EOxTimecontrol._esm_template.replace(
        "view.el.appendChild(el);", time_control_element
    )

# ------------------------------------------------------------------------------
# Create and Display Map & Widgets
# ------------------------------------------------------------------------------
# 1. Map widget. The id lets the layer control and time control find this map (`for_`).
#    `center` is [lon, lat]. Widgets need an explicit height, otherwise the output collapses.
eox_map = EOxMap(
    id="main-map",
    center=[15, 45],
    zoom=4,
    layout=Layout(width="100%", height="500px"),
)
eox_map.layers = layers

# 2. Timecontrol widget at the bottom of the map. Starts on the first date of the cube.
time_control = EOxTimecontrol(
    for_="#main-map",
    init_date=["first"],
    layout=Layout(width="100%"),
)

# 3. Layercontrol panel on the left side. Only the "config" (variable, min/max) and "opacity"
#    tools are shown; 'datetime' is left out because the time control below handles time.
#    (`for_` because `for` is a Python keyword.)
control = EOxLayercontrol(
    for_="#main-map",
    tools=["config", "opacity"],
    layout=Layout(flex="1", height="550px", width="300px", min_width="250px"),
)

# 4. Right container: Map on top, Timecontrol on bottom
map_container = VBox(
    [eox_map, time_control],
    layout=Layout(flex="2", display="flex", flex_direction="column", gap="10px"),
)

# 5. Full UI container: Layercontrol on left, Map + Timecontrol on right
HBox(
    [control, map_container],
    layout=Layout(width="100%", display="flex", gap="16px", align_items="flex-start"),
)



## 2. Antarctic sea-ice cube in a polar projection (EPSG:6932)

The second map shows [`sea-ice-cube-geozarr-v2.zarr`](https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/sea-ice-cube/sea-ice-cube-geozarr-v2.zarr/zarr.json): 146 time steps from 2011-02-01, on a 712 × 712 grid of 12.5 km pixels. It is stored in **EPSG:6932** (EASE-Grid 2.0 South, a Lambert azimuthal equal-area projection centred on the South Pole) and declares this with `"proj:code": "EPSG:6932"`. Here we show `sss` (sea surface salinity).

Polar data drawn in Web Mercator, the default for web maps, is badly stretched and cut off near the pole. So this map uses the data's own projection. That takes a few extra steps:

1. **Register the projection.** OpenLayers only knows `EPSG:4326` and `EPSG:3857` out of the box, so we give it the [proj4](https://epsg.io/6932) definition and the valid extent of EPSG:6932 with `register_projection()`.
2. **Use a basemap in the same projection.** The WMTS settings in `options.json` point to an Antarctic basemap that uses EPSG:6932 tiles.
3. **Set the map projection before adding layers.** That way both layers are drawn in EPSG:6932 directly.

Some details in the code below are workarounds for how the Jupyter widgets behave. Each is explained in a code comment:

- `register_projection()` is sent through the map from section 1. A widget only receives Python calls once it is shown, and registered projections are shared by all maps on the page.
- `"projection": MAP_CRS` is also set on the GeoZarr source, because the source can't find projections registered on the map by itself.
- The map starts at lon/lat `[0, -80]` (Antarctica) before switching to EPSG:6932. Starting at the default `[0, 0]` (the equator) would place the view outside the area EPSG:6932 can show.

In [2]:
# ==============================================================================
# 2. Antarctic sea-ice cube: GeoZarr in a polar projection (EPSG:6932)
# ==============================================================================
import os
import json


# EASE-Grid 2.0 South: Lambert azimuthal equal-area centred on the South Pole, in metres.
# The proj4 string comes from https://epsg.io/6932 and the extent is the valid area
# [minx, miny, maxx, maxy]. It matches the store's `spatial:bbox`.
MAP_CRS = "EPSG:6932"
MAP_PROJ4 = "+proj=laea +lat_0=-90 +lon_0=0 +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs +type=crs"
MAP_EXTENT = [-4450000, -4450000, 4450000, 4450000]

# GeoZarr store in EPSG:6932 (`"proj:code": "EPSG:6932"` in its zarr.json), 3 resolution levels
polar_zarr_url = "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/sea-ice-cube/sea-ice-cube-geozarr-v2.zarr"

# Load local basemap WMTS options: tile URL template, layer name, tile matrix set, and the
# EPSG:6932 tile grid (origins + resolutions) of the Antarctic basemap
opts_file = "./options.json" if os.path.exists("./options.json") else "options.json"
with open(opts_file, "r") as f:
    options = json.load(f)

layers = [
    # Basemap: WMTS tiles that are already in EPSG:6932, so no reprojection is needed
    {
        "type": "Tile",
        "properties": {
            "id": "polar_basemap",
            "title": "Polar Basemap",
        },
        "source": {
            "type": "WMTS",
            "projection": "EPSG:6932",
            **options
        },
    },
     {
        "type": "WebGLTile",
        "properties": {
            "id": "basel-melt",
            "title": "Basel Melt GeoZarr",
            "layerControlExpand": True,
            "layerControlToolsExpand": True,
            # Layer control form (Variable dropdown + Min / Max), filled from the store metadata
            "layerConfig": {
                "type": "style",
                "style": True,
                "autofill": True,
            },
        },
        "source": {
            "type": "GeoZarr",
            "url": polar_zarr_url,
            # The GeoZarr source looks up the store's `proj:code` in its own projection
            # registry, which doesn't see `register_projection()`. Passing the projection here
            # lets <eox-map> resolve it from the registry EPSG:6932 was registered in.
            "projection": MAP_CRS,
            # Start on sea surface salinity
            "bands": ["sss"],
        },
        # Same colour ramp as the Mediterranean map: band 1 = value, band 2 = data mask (alpha)
        "style": {
                    "variables": {
                        "min": 0,
                        "max": 40,
                    },
                    "color": [
                        "interpolate",
                        ["linear"],
                        ["band", 1],
                        ["var", "min"],
                        ["color", 0, 210, 255, ["band", 2]],
                        [
                            "+",
                            ["var", "min"],
                            ["*", ["-", ["var", "max"], ["var", "min"]], 0.3333333333333333],
                        ],
                        ["color", 254, 217, 118, ["band", 2]],
                        [
                            "+",
                            ["var", "min"],
                            ["*", ["-", ["var", "max"], ["var", "min"]], 0.6666666666666666],
                        ],
                        ["color", 253, 141, 60, ["band", 2]],
                        ["var", "max"],
                        ["color", 189, 0, 38, ["band", 2]],
                    ],
                },
    },
]


# ------------------------------------------------------------------------------
# Create and Display Map & Widgets
# ------------------------------------------------------------------------------
# Register EPSG:6932 through the map from the first cell, which is already rendered: a widget
# only receives messages once it is on screen, and registered projections are shared by every
# <eox-map> on the page, so the polar map below can use EPSG:6932 as soon as it renders.
# (Run section 1 first, and keep its output visible.)
eox_map.register_projection(MAP_CRS, MAP_PROJ4, MAP_EXTENT)

# 1. Map widget in the polar projection. The widget applies `config` (the polar view) before
# `layers`, so EPSG:6932 is the map projection by the time the layers are added. Keep the layers
# out of `config`: the `layers` argument defaults to [] and would overwrite them.
eox_map = EOxMap(
    id="polar-map",
    # Start over Antarctica (lon/lat): switching from Web Mercator [0, 0] (equator) to the
    # polar projection would give a view outside EPSG:6932's valid area -> NaN coordinates.
    center=[0, -80],
    zoom=1,
    # Switch the map to EPSG:6932. The center above is carried over into polar coordinates.
    config={"view": {"projection": MAP_CRS}},
    layers=layers,
    layout=Layout(width="100%", height="500px"),
)

# 2. Timecontrol widget at the bottom of the map (dates from the store's `time` dimension)
time_control = EOxTimecontrol(
    for_="#polar-map",
    init_date=["first"],
    layout=Layout(width="100%"),
)

# 3. Layercontrol panel on the left side (exclude 'datetime' to avoid duplicate time controls)
control = EOxLayercontrol(
    for_="#polar-map",
    tools=["config", "opacity"],
    layout=Layout(flex="1", height="550px", width="300px", min_width="250px"),
)

# 4. Right container: Map on top, Timecontrol on bottom
map_container = VBox(
    [eox_map, time_control],
    layout=Layout(flex="2", display="flex", flex_direction="column", gap="10px"),
)

# 5. Full UI container: Layercontrol on left, Map + Timecontrol on right
HBox(
    [control, map_container],
    layout=Layout(width="100%", display="flex", gap="16px", align_items="flex-start"),
)

## Things to try

- **Switch variables** in the layer control (*Variable* dropdown), or start on one directly with `"bands": ["to"]` in the Mediterranean layer.
- **Change the colour range** with the *Min / Max* inputs, or the `min` / `max` values under `style.variables`. Only the GPU colouring changes; no data is downloaded again.
- **Move through time** with the time control under each map. Each date shows a different `time` slice of the same store.
- **Change the colours** by editing the `["color", r, g, b, alpha]` stops in the style. For example, add more stops to get a finer colour ramp.
- **Show your own store**: any store that follows the GeoZarr conventions above (e.g. the output of `2_geozarr_conversion.ipynb`) can be used as `url`. If it's not in EPSG:4326 or EPSG:3857, register its projection as in section 2.